## Task 2 Notebook

- Load `telemetry_raw.csv`, handling the thousands-separator formatting (e.g. `"12,500"`) via `thousands=","`
- Parse timestamps as day-first (`DD/MM/YYYY`) — an assumption worth confirming with whoever owns the export
- Validate the raw data against a small set of rules (no duplicates, and `timestamp`/`active_power`/`setpoint` all present), then keep only rows that pass
- Re-run the same validation on the cleaned output as a safety check, and stop before saving if anything still fails
- Save the valid rows to `telemetry_cleaned.csv`

In [1]:
import pandas as pd

RAW_PATH = "telemetry_raw.csv"
CLEAN_PATH = "telemetry_cleaned.csv"

In [2]:
raw = pd.read_csv(RAW_PATH, thousands=",")
print(f"{len(raw)} rows loaded")
raw

12 rows loaded


,timestamp,active_power,setpoint,site_id
0,01/07/2026 00:00:00,18200.0,18000.0,1
1,01/07/2026 00:30:00,18450.0,18100.0,1
2,01/07/2026 01:00:00,12500.0,18300.0,1
3,01/07/2026 01:30:00,19000.0,NaN,1
4,01/07/2026 02:00:00,19100.0,18800.0,1
5,01/07/2026 02:30:00,19350.0,18950.0,1
6,01/07/2026 02:30:00,19350.0,18950.0,1
7,01/07/2026 03:00:00,NaN,19100.0,1
8,01/07/2026 03:30:00,19700.0,19250.0,1
9,01/07/2026 04:00:00,19850.0,19300.0,1


In [3]:
df = raw.copy()
df["timestamp"] = pd.to_datetime(df["timestamp"], dayfirst=True, errors="coerce")

In [4]:
def validate(frame, keep):
    return {
        "no duplicates": ~frame.duplicated(keep=keep),
        "has timestamp": frame["timestamp"].notna(),
        "has active_power": frame["active_power"].notna(),
        "has setpoint": frame["setpoint"].notna(),
    }


def report(rules):
    all_passed = True
    for name, passed in rules.items():
        fail_count = (~passed).sum()
        status = "PASS" if fail_count == 0 else "FAIL"
        all_passed &= fail_count == 0
        print(f"  [{status}] {name}: {fail_count} row(s) failed")
    return all_passed


print("Validation results pre-cleanse:")
pre_rules = validate(df, keep="first")
report(pre_rules)

valid = pd.concat(pre_rules, axis=1).all(axis=1)
clean = df[valid].sort_values("timestamp").reset_index(drop=True)
print(f"\n{len(df)} rows in -> {len(clean)} valid, {(~valid).sum()} invalid (dropped)")

print("\nValidation results post-cleanse:")
post_rules = validate(clean, keep=False)
assert report(post_rules), "Cleaned data failed post-validation — not safe to load"
print("\nAll checks passed — safe to load.")
clean

Validation results pre-cleanse:
  [FAIL] no duplicates: 1 row(s) failed
  [FAIL] has timestamp: 1 row(s) failed
  [FAIL] has active_power: 1 row(s) failed
  [FAIL] has setpoint: 1 row(s) failed

12 rows in -> 8 valid, 4 invalid (dropped)

Validation results post-cleanse:
  [PASS] no duplicates: 0 row(s) failed
  [PASS] has timestamp: 0 row(s) failed
  [PASS] has active_power: 0 row(s) failed
  [PASS] has setpoint: 0 row(s) failed

All checks passed — safe to load.


,timestamp,active_power,setpoint,site_id
0,2026-07-01 00:00:00,18200.0,18000.0,1
1,2026-07-01 00:30:00,18450.0,18100.0,1
2,2026-07-01 01:00:00,12500.0,18300.0,1
3,2026-07-01 02:00:00,19100.0,18800.0,1
4,2026-07-01 02:30:00,19350.0,18950.0,1
5,2026-07-01 03:30:00,19700.0,19250.0,1
6,2026-07-01 04:00:00,19850.0,19300.0,1
7,2026-07-01 04:30:00,20500.0,19550.0,1


In [5]:
clean.to_csv(CLEAN_PATH, index=False)
print(f"Saved {len(clean)} rows to {CLEAN_PATH}")

Saved 8 rows to telemetry_cleaned.csv


## Summary

Loaded the raw telemetry export, fixed a thousands-separator formatting
issue, and validated every row against four rules (no duplicates, and
`timestamp`/`active_power`/`setpoint` all present). 4 of 12 rows failed
validation and were dropped. The same four rules were then re-checked
against the cleaned output — all passed — before saving 8 clean rows to
`telemetry_cleaned.csv`.